# COVID-19 Single-Cell RNA-seq Analysis

## Overview

This notebook demonstrates a typical single-cell RNA-seq analysis workflow, including:

- **Data pre-processing**: quality control metrics, gene and cell filtering  
- **Dimensionality reduction & clustering** to explore the dataset structure  
- **Visualization** of key features  

### Note on Data Efficiency

The computationally intensive downstream analyses (clustering, UMAP) are performed on a reproducible **25,000-cell subset** to ensure consistent results across runs.

## Note on the dataset and workflow

I start from pre-normalized data from the dataset and recompute key metrics



In [ ]:
# Imports and plotting setup

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

sc.settings.verbosity = 1

plt.style.use("default")

sc.set_figure_params(
    dpi=100,
    facecolor="white",
    figsize=(7, 5),
)

# Some parameters to reduce the dataset size so it fit in my PC ram.
RANDOM_SEED = 42
N_CELLS = 25_000

np.random.seed(RANDOM_SEED)

## 1. Load the dataset


In [ ]:
path = "../datasets/fe2e847c-1602-4f1b-86a4-112e4dc7a8e3.h5ad"

adata = sc.read_h5ad(path)

adata.obs_names_make_unique()
adata.var_names_make_unique()

print(adata)
print(f"Cells: {adata.n_obs:,}")
print(f"Genes: {adata.n_vars:,}")

sample_counts = adata.obs["sample_id"].value_counts()

print(f"Number of samples: {sample_counts.size}")
print(sample_counts.describe())

rng = np.random.default_rng(RANDOM_SEED)

cells_per_sample = max(1, N_CELLS // sample_counts.size)

selected_cells = []

for sample_id, n_cells in sample_counts.items():

    sample_indices = np.flatnonzero(adata.obs["sample_id"].values == sample_id)

    n_select = min(cells_per_sample, len(sample_indices))

    selected = rng.choice(
        sample_indices,
        size=n_select,
        replace=False,
    )

    selected_cells.extend(selected)

selected_cells = np.asarray(selected_cells)

print(f"Selected cells: {len(selected_cells):,}")

adata_small = adata[selected_cells].to_memory()
print(adata_small)
print(f"RAM dataset: {adata_small.n_obs:,} cells × {adata_small.n_vars:,} genes")
adata.file.close()
del adata

## 2. Check the expression matrix

The matrix in this file is already transformed. It is not raw integer UMI counts.

Therefore this notebook does **not** run `normalize_total()` or `log1p()` again.


In [ ]:
X = adata_small.X

if hasattr(X, "data"):
    nonzero_values = X.data
else:
    nonzero_values = np.asarray(X).ravel()

print("Type:", type(X))
print("Shape:", X.shape)
print("Dtype:", X.dtype)
print("Minimum non-zero value:", nonzero_values.min())
print("Maximum value:", nonzero_values.max())

integer_fraction = np.mean(nonzero_values == np.round(nonzero_values))
print(f"Fraction of non-zero values that are integers: {integer_fraction:.6f}")

## 3. Identify mitochondrial genes


In [ ]:
# var_names are Ensembl IDs in this dataset.
# Gene symbols are stored in feature_name.

adata_small.var["gene_symbol"] = adata_small.var["feature_name"].astype(str)

adata_small.var["mt"] = adata_small.var["gene_symbol"].str.upper().str.startswith("MT-")

print("Mitochondrial genes:", int(adata_small.var["mt"].sum()))

adata_small.var.loc[adata_small.var["mt"], "gene_symbol"].head(30).tolist()

## 4. Calculate QC metrics


In [ ]:
sc.pp.calculate_qc_metrics(
    adata_small,
    qc_vars=["mt"],
    inplace=True,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(
    adata_small.obs["n_genes_by_counts"],
    bins=100,
)
axes[0].axvline(200, linestyle="--")
axes[0].set_xlabel("Detected genes")
axes[0].set_ylabel("Cells")

axes[1].hist(
    adata_small.obs["pct_counts_mt"],
    bins=100,
)
axes[1].axvline(10, linestyle="--")
axes[1].set_xlabel("Mitochondrial expression (%)")
axes[1].set_ylabel("Cells")

plt.tight_layout()

## 5. Cell QC

- keep cells with at least 200 detected genes
- remove cells with more than 10% mitochondrial expression

We do not use `total_counts` as a raw sequencing-depth filter because the matrix is already transformed.


In [ ]:
cell_qc = (adata_small.obs["n_genes_by_counts"] >= 200) & (
    adata_small.obs["pct_counts_mt"] < 10
)

print(f"Cells before QC: {adata_small.n_obs:,}")
print(f"Cells after QC:  {cell_qc.sum():,}")
print(f"Cells removed:   {(~cell_qc).sum():,}")

adata_small_qc = adata_small[cell_qc]

## 6. Filter rarely detected genes


In [ ]:
genes_before = adata_small_qc.n_vars

# keep the genes express in at least 10 cells
sc.pp.filter_genes(
    adata_small_qc,
    min_cells=10,
)

# keep the genes whose expression value is >= 1
sc.pp.filter_genes(adata_small_qc, min_counts=1)

genes_after = adata_small_qc.n_vars

print(f"Genes before filtering: {genes_before:,}")
print(f"Genes after filtering:  {genes_after:,}")
print(f"Genes removed:         {genes_before - genes_after:,}")

## 7. Create a manageable cell-level dataset


In [ ]:
# The full dataset is too large for comfortable local PCA/UMAP analysis.
# Use a reproducible subset for the cell-level workflow.

N_CELLS = 24_000

rng = np.random.default_rng(RANDOM_SEED)

if adata_small_qc.n_obs > N_CELLS:
    selected = rng.choice(
        adata_small_qc.n_obs,
        size=N_CELLS,
        replace=False,
    )
    adata_analysis = adata_small_qc[selected].copy()
else:
    adata_analysis = adata_small_qc.copy()

print(adata_analysis)

## 8. Highly variable genes


In [ ]:
sc.pp.highly_variable_genes(
    adata_analysis,
    n_top_genes=2_000,
    flavor="seurat",
)

print("Highly variable genes:", int(adata_analysis.var["highly_variable"].sum()))

In [ ]:
# Keep only HVGs for PCA and the neighborhood graph.

adata_analysis = adata_analysis[:, adata_analysis.var["highly_variable"]].copy()

print(adata_analysis)

## 9. PCA


In [ ]:
sc.tl.pca(
    adata_analysis,
    n_comps=30,
    svd_solver="arpack",
    random_state=RANDOM_SEED,
)

sc.pl.pca_variance_ratio(
    adata_analysis,
    n_pcs=30,
)

In [ ]:

# Check the different sites to see if there are batches or sampling differences that needs to be taken into account
sc.pl.pca(
    adata_analysis,
    color="Site",
    size=8,
    alpha=0.5,
)

## 10. UMAP and cell populations


In [ ]:
sc.pp.neighbors(
    adata_analysis,
    n_neighbors=15,
    n_pcs=30,
)

sc.tl.umap(
    adata_analysis,
    random_state=RANDOM_SEED,
)

In [ ]:
sc.pl.umap(
    adata_analysis,
    color="cell_type",
    size=8,
    alpha=0.6,
)

# Can it be interesting to see what clusters can be better pulled apart with Bonsai (newer method than UMAP)

In [ ]:
# Clinical status on the same embedding.
# This is descriptive, not the main statistical analysis but can see that a subgroup of critical patient are very close to each other at the top and they formed a well distinct cluster.

sc.pl.umap(
    adata_analysis,
    color="Status_on_day_collection_summary",
    size=8,
    alpha=0.5,
)

## 11. Cell-type composition per sample


In [ ]:
# Work from the full QC-filtered metadata.
# No expression matrix is needed for this analysis.

composition = pd.crosstab(
    adata_small_qc.obs["sample_id"],
    adata_small_qc.obs["cell_type"],
    normalize="index",
)

print(f"Samples: {composition.shape[0]} | " f"Cell types: {composition.shape[1]}")

composition.head()

In [ ]:
# Show the composition of each sample 
# That just usefull if a patient ask for their data

ax = composition.plot(
    kind="bar",
    stacked=True,
    figsize=(60, 5),
    legend=False,
)

ax.set_xlabel("Sample")
ax.set_ylabel("Fraction of cells")
ax.set_title("Cell-type composition by sample")

plt.tight_layout()

Each sample is now one observation. This is more useful for clinical comparisons than plotting every cell.


## 12. Cell composition by clinical status


In [ ]:
sample_info = (
    adata_small_qc.obs[
        [
            "sample_id",
            "Status_on_day_collection_summary",
            "Site",
            "donor_id",
        ]
    ]
    .drop_duplicates("sample_id")
    .set_index("sample_id")
)

composition_status = composition.join(sample_info)

status_means = composition_status.groupby("Status_on_day_collection_summary").mean(
    numeric_only=True
)

status_means    

In [ ]:
# Set figure dimensions
plt.figure(figsize=(20, 20))

# Generate heatmap
sns.heatmap(
    status_means,
    annot=False,          # Display numeric mean values inside cells
    fmt=".2f",           # Format values to 2 decimal places
    cmap="coolwarm",       # Palette (e.g., 'viridis', 'coolwarm', 'vlag', 'Blues')
    linewidths=0.5,      # Grid lines separating cells
    cbar_kws={"label": "Mean Proportions"}
)

plt.title("Mean Cell Composition by Status", fontsize=14, pad=12)
plt.xlabel("Cell Type / Variable", fontsize=11)
plt.ylabel("Status", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
ax = status_means.plot(
    kind="bar",
    stacked=True,
    figsize=(11, 5),
    legend=True,
)

ax.set_xlabel("Clinical status")
ax.set_ylabel("Mean fraction of cells per sample")
ax.set_title("Average cell-type composition by clinical status")

box = ax.get_position()
ax.set_position([box.x0, box.y0 + box.height * 0.1, box.width, box.height * 0.9])

# Put a legend below current axis
ax.legend(
    loc="upper center", bbox_to_anchor=(0.5, -0.5), fancybox=True, shadow=True, ncol=5
)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()

The previous plot asks whether the overall cellular composition differs between clinical groups. Maybe Like in the study it is better to sort the cell types and make multiple plots for the subgroups of cell types.

The next plot shows the major cell types separately, so large differences are easier to see.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

composition_long = (
    composition.reset_index()
    .melt(
        id_vars="sample_id",
        var_name="cell_type",
        value_name="fraction",
    )
    .merge(
        sample_info[["Status_on_day_collection_summary"]].reset_index(),
        on="sample_id",
        how="left",
    )
)

major_cell_types = (
    composition_long.groupby("cell_type")["fraction"]
    .mean()
    .sort_values(ascending=False)
    .head(12)
    .index
)

plot_data = composition_long[
    composition_long["cell_type"].isin(major_cell_types)
].copy()


g = sns.catplot(
    data=plot_data,
    x="Status_on_day_collection_summary",
    y="fraction",
    col="cell_type",
    col_wrap=4,
    kind="box",
    sharey=False,
    sharex=False,  
    height=3.5,
    aspect=1.2,
)

# Adjust axis labels and reduce title font size to prevent cell type title collisions
g.set_axis_labels("", "Fraction of cells")
g.set_titles("{col_name}", size=10, weight="bold")

# Explicitly rotate and display x-tick labels for every facet
for ax in g.axes.flat:
    ax.tick_params(
        labelbottom=True, labelsize=8
    )  # Ensures x-labels are visible on all rows
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Add generous margin padding around subplots
# hspace controls vertical spacing between rows (prevents title colliding with upper x-axis)
# wspace controls horizontal spacing between columns
g.figure.subplots_adjust(hspace=0.6, wspace=0.35, bottom=0.25, top=0.92)

plt.show()

There are some major differences with the naive B cell for the LPS subgroups. Those plots help us see that our heatmap was indeed usefull to display a summary view on the major differences between the patient status.

## 13. Sample-level PCA


Here the observations are samples, not cells.

Two samples that are close together have similar cell-type composition.


In [ ]:
X_composition = composition.values

# Center the cell-type proportions.
X_centered = X_composition - X_composition.mean(axis=0)

sample_pca_model = PCA(
    n_components=2,
    random_state=RANDOM_SEED,
)

sample_coords = sample_pca_model.fit_transform(X_centered)

sample_pca = pd.DataFrame(
    sample_coords,
    index=composition.index,
    columns=["PC1", "PC2"],
).join(sample_info)

print("Variance explained:", sample_pca_model.explained_variance_ratio_)

sample_pca.head()

In [ ]:
plt.figure(figsize=(8, 6))

ax = sns.scatterplot(
    data=sample_pca,
    x="PC1",
    y="PC2",
    hue="Status_on_day_collection_summary",
    style="Site",
    s=90,
)

plt.axhline(0, linestyle="--", linewidth=0.5)
plt.axvline(0, linestyle="--", linewidth=0.5)

box = ax.get_position()
ax.set_position([box.x0, box.y0 + box.height * 0.1, box.width, box.height * 0.9])

# Put a legend below current axis
ax.legend(
    loc="upper center", bbox_to_anchor=(0.5, -0.1), fancybox=True, shadow=True, ncol=5
)


plt.title("Samples represented by cell-type composition")
plt.tight_layout()

This plot is easier to interpret than the cell-level PCA for the clinical question:

- each point = one sample
- color = clinical status
- marker = site

We do not see a very clear separation between sample on the status and the site.


## 14. Distances between samples


The goal here is to measure how similar samples are using their cell-type proportions.

Bray-Curtis distance is useful for compositional data: a small value means similar composition, while a large value means different composition.


In [ ]:
distance_array = pairwise_distances(
    composition.values,
    metric="braycurtis",
)

distance_matrix = pd.DataFrame(
    distance_array,
    index=composition.index,
    columns=composition.index,
)

distance_matrix.iloc[:5, :5]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


g = sns.clustermap(
    distance_matrix,
    cmap="viridis",
    xticklabels=False,
    yticklabels=False,
    figsize=(10, 8),
    # Optional: metric="euclidean", method="average"
)

g.fig.suptitle("Clustered Distance Between Samples (Cell Composition)", y=1.02)
g.ax_heatmap.set_xlabel("Sample")
g.ax_heatmap.set_ylabel("Sample")

plt.show()

## 15. Within-status vs between-status distances


In [ ]:
groups = sample_info["Status_on_day_collection_summary"]

within_distances = []
between_distances = []

samples = distance_matrix.index

for i, sample_a in enumerate(samples):
    for j in range(i + 1, len(samples)):
        sample_b = samples[j]

        group_a = groups.loc[sample_a]
        group_b = groups.loc[sample_b]

        if pd.isna(group_a) or pd.isna(group_b):
            continue

        distance = distance_matrix.loc[sample_a, sample_b]

        if group_a == group_b:
            within_distances.append(distance)
        else:
            between_distances.append(distance)

distance_summary = pd.DataFrame(
    {
        "comparison": (
            ["Same clinical status"] * len(within_distances)
            + ["Different clinical status"] * len(between_distances)
        ),
        "distance": within_distances + between_distances,
    }
)

print(distance_summary.groupby("comparison")["distance"].describe())

In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=distance_summary,
    x="comparison",
    y="distance",
)

sns.stripplot(
    data=distance_summary,
    x="comparison",
    y="distance",
    color="black",
    alpha=0.25,
    size=3,
)

plt.xlabel("")
plt.ylabel("Bray-Curtis distance")
plt.title("Sample distances by clinical status")

plt.xticks(rotation=15)
plt.tight_layout()

As expected by the results of the previous clustering plots the differences between two samples with the same status and other status is not clear. 


## 16. Save the small analysis object


In [ ]:
output_path = "../datasets/covid_scRNA_analysis_20k.h5ad"

adata_analysis.write(output_path)

print(f"Saved: {output_path}")